# tp-05m 多層スタッキング（L2 メタ学習器 + L3 平均）の比較

現行のアンサンブル v2 は 1 層（logit 空間の重み付き平均 / 前進選択）。Kaggle の上位解法でよく使われる**多層スタッキング**（L1 = 各モデルの OOF、L2 = メタ学習器、L3 = 平均）を実装して比べる。
L1 の OOF（valid 2022〜2024）を使うので追加の学習は小さい。メタ学習器は leave-one-year-out（他の年の OOF だけで当てはめて当該年に適用）で評価する。

## 設計
- **L1 の候補**: M7 = v2 の 7 モデル / M+ = 全 OOF（調整後 LightGBM・XGBoost・NN、CatBoost、素の LightGBM・XGBoost、ランキング系の較正版、専用 weighted があれば）。
- **L2 のメタ学習器**: (a) 既存のロジスティック・スタッキング（参照）、(b) 浅い LightGBM（メンバーの logit + 平均・ばらつき・レース内の位置・頭数、他の目的の平均 logit）、(c) 小さな NN（レース内の文脈つき listwise softmax + BCE。入力は (b) と同じ）。
- **L3**: L2 出力と v2 の logit 平均（重みを当てはめない = 入れ子の LOYO を避ける）。
- 採用基準 = v2 との LogLoss の差の 95% 信頼区間の上限 < 0 かつ連対・複勝が悪化しない（0.0002 以内）。test 2025 は使わない。

In [ ]:
import os, sys
from pathlib import Path


def _find_root():
    p = Path.cwd().resolve()
    for q in [p, *p.parents]:
        if (q / "AGENTS.md").exists() and (q / "src").is_dir():
            return q
    raise RuntimeError("リポジトリルートが見つかりません")


REPO_ROOT = _find_root()
os.chdir(REPO_ROOT)
os.environ.setdefault("POLARS_MAX_THREADS", "4")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.utils.project_env import load_project_dotenv
load_project_dotenv()

In [ ]:
import json, subprocess, time
import numpy as np
import pandas as pd
import mlflow

from src.pipeline.mlflow.train_utils import require_tracking_server
from src.pipeline.models import ensemble as en, stacking as st, tp05_data, win_models as wm

WORK = REPO_ROOT / "data/local/modeling/training_pipeline"
OOF_DIR = WORK / "oof/model_oof"
OUT = WORK / "improve_m"
OUT.mkdir(parents=True, exist_ok=True)
THREADS = int(os.environ.get("TP05M_THREADS", "4"))
mlflow.set_experiment("keiba-tp-ensemble")
require_tracking_server()
GIT_COMMIT = subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True, text=True, cwd=REPO_ROOT).stdout.strip()
D = tp05_data.load_dataset(WORK)
meta = D["meta"]
key = ["race_id", "horse_id"]
v2 = json.loads((WORK / "ensemble_v2/ensemble_spec.json").read_text(encoding="utf-8"))
M7 = v2["models"]
EXTRA = ["lgb_tuned", "xgb_tuned", "nn_tuned", "cat_binary", "lgb_binary", "xgb_binary", "lambdarank_calibrated", "xendcg_calibrated", "lgb_course_weighted", "lgb_grade_weighted", "cat_extras_ped", "lgb_dart_extras_ped", "lgb_et_extras_ped", "xgb_deep_extras_ped", "nn_tabm_extras_ped", "nn_noise_extras_ped"]
MPLUS = M7 + [n for n in EXTRA if (OOF_DIR / f"{n}.parquet").exists()]
print("M7:", M7); print("M+:", MPLUS)
base = meta[key + ["finish_position", "year"]].merge(pd.read_parquet(OOF_DIR / f"{M7[0]}.parquet")[key], on=key, how="inner")
for n in MPLUS:
    o = pd.read_parquet(OOF_DIR / f"{n}.parquet")[key + ["p_win", "p_top2", "p_top3"]].rename(columns={f"p_{t}": f"{n}__{t}" for t in ("win", "top2", "top3")})
    base = base.merge(o, on=key, how="inner")
v2oof = base[key].merge(pd.read_parquet(OOF_DIR / "ensemble_v2.parquet")[key + ["p_win", "p_top2", "p_top3"]], on=key, how="left")
race, years, fin = base["race_id"].to_numpy(), base["year"].to_numpy(), base["finish_position"].to_numpy()
Y = {"win": (fin == 1).astype(float), "top2": (fin <= 2).astype(float), "top3": (fin <= 3).astype(float)}
TOT = st.TOT
print(f"行 {len(base):,}")

In [ ]:
def build_F(names):
    Zt = {t: en.logit(np.clip(np.stack([base[f"{n}__{t}"].to_numpy() for n in names], axis=1), 1e-6, 1 - 1e-6)) for t in TOT}
    mean_all = {t: Zt[t].mean(1) for t in TOT}
    F = {}
    for t in TOT:
        f = st.meta_features(Zt[t], race, names)
        for u in TOT:
            if u != t:
                f[f"zmean_{u}"] = mean_all[u]
        F[t] = f
    Fall = pd.concat([st.meta_features(Zt[t], race, names, with_context=(t == "win")).add_suffix(f"__{t}") for t in TOT], axis=1)
    return Zt, F, Fall


def cmp(P, name):
    out = {"variant": name}
    for t in TOT:
        b = wm.paired_bootstrap_diff(wm.per_race_loss(race, v2oof[f"p_{t}"].to_numpy(), Y[t]), wm.per_race_loss(race, P[t], Y[t]), n_boot=1000)
        out[f"{t}_v2"], out[f"{t}_new"], out[f"Δ{t}"], out[f"{t}_hi"] = en.ll(v2oof[f"p_{t}"].to_numpy(), Y[t]), en.ll(P[t], Y[t]), b["diff"], b["hi"]
    out["採用"] = bool(out["win_hi"] < 0 and out["Δtop2"] <= 2e-4 and out["Δtop3"] <= 2e-4)
    return out


V2P = {t: v2oof[f"p_{t}"].to_numpy() for t in TOT}
ROWS, STORE = [], {}
for pool_name, names in (("M7", M7), ("M+", MPLUS)):
    t0 = time.time()
    Zt, F, Fall = build_F(names)
    # L2(a): 既存のロジスティック・スタッキング（参照）
    La = {}
    for t in TOT:
        _, La[t] = en.loyo_eval("stack_lr", en.sigmoid(Zt[t]), years, race, Y[t], TOT[t])
    # L2(b): 浅い LightGBM
    Lb = st.loyo_lgb_meta({t: F[t].to_numpy() for t in TOT}, Y, years, race, threads=THREADS)
    # L2(c): 小さな NN
    Lc = st.loyo_nn_meta(Fall, Y, years, race)
    # L2(d)/(e): 条件付きロジット（勝率はレース内 softmax、重み自由）。(e) は 3 つの目的の logit を互いの入力にする（クロスターゲット）
    Ld = st.loyo_clogit_meta(Zt, Y, years, race)
    Le = st.loyo_clogit_meta(Zt, Y, years, race, cross=True)
    L3 = {t: st.logit_mean([V2P[t], Lb[t], Lc[t]], race, TOT[t]) for t in TOT}
    L3d = {t: st.logit_mean([V2P[t], Ld[t], Le[t]], race, TOT[t]) for t in TOT}
    L3e = {t: st.logit_mean([V2P[t], Ld[t], Le[t], Lb[t], Lc[t]], race, TOT[t]) for t in TOT}
    L3b = {t: st.logit_mean([V2P[t], Lb[t]], race, TOT[t]) for t in TOT}
    L3c = {t: st.logit_mean([V2P[t], La[t], Lb[t], Lc[t]], race, TOT[t]) for t in TOT}
    for nm, P in ((f"{pool_name}: L2 ロジスティック", La), (f"{pool_name}: L2 条件付きロジット", Ld), (f"{pool_name}: L2 条件付きロジット(クロスターゲット)", Le), (f"{pool_name}: L2 LightGBM", Lb), (f"{pool_name}: L2 NN", Lc),
                  (f"{pool_name}: L3 = v2 + 条件付きロジット2種", L3d), (f"{pool_name}: L3 = v2 + 全 L2（条件付き2・LightGBM・NN）", L3e), (f"{pool_name}: L3 = v2 + L2 LightGBM + L2 NN", L3),
                  (f"{pool_name}: L3 = v2 + L2 LightGBM", L3b), (f"{pool_name}: L3 = v2 + 3 つの L2", L3c)):
        ROWS.append(cmp(P, nm)); STORE[nm] = P
    print(pool_name, f"{time.time() - t0:.0f}s", flush=True)
R = pd.DataFrame(ROWS).set_index("variant")
display(R[["win_v2", "win_new", "Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5)); R.to_csv(OUT / "stacking_vs_v2.csv")

In [ ]:
best_name = R.sort_values("Δwin").index[0]
print("勝率で最良:", best_name, R.loc[best_name, ["Δwin", "win_hi", "Δtop2", "Δtop3", "採用"]].round(5).to_dict())
for nm in R.index[R["採用"]]:
    o = base[key + ["year"]].copy()
    for t in TOT:
        o[f"p_{t}"] = STORE[nm][t]
    o.to_parquet(OOF_DIR / ("stack_" + nm.replace(": ", "_").replace(" ", "").replace("=", "").replace("+", "_").replace("/", "") + ".parquet"), index=False)
(OUT / "report.json").write_text(json.dumps({"git_commit": GIT_COMMIT, "M7": M7, "M_plus": MPLUS, "results": R.round(6).reset_index().to_dict("records"), "best": best_name,
                                             "adopted": list(R.index[R["採用"]])}, ensure_ascii=False, indent=1, default=str))
with mlflow.start_run(run_name="tp-05m-multilayer-stacking", tags={"notebook": "tp-05m", "git_commit": GIT_COMMIT}):
    for n, r in R.iterrows():
        mlflow.log_metrics({("delta_win__" + n).replace(" ", "").replace(":", "_").replace("+", "_").replace("=", "")[:240]: float(r["Δwin"])})
    for f_ in ("stacking_vs_v2.csv", "report.json"):
        mlflow.log_artifact(str(OUT / f_), "improve_m")